# Validação do ETL

Vou verificar se a base consolidada pelo ETL está pronta para análise. O foco aqui é confirmar a estrutura da tabela, a tipagem das colunas, a completude por ano e as regras de negócio do desafio, como a distinção entre função, subfunção e linhas agregadas.

Também vou registrar um ponto importante desde já: o ano de 2025 está parcial na base, então qualquer comparação temporal precisa considerar essa limitação.

In [1]:
# Importação das bibliotecas
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Configuração da leitura

Antes da validação, defino a apresentação dos números em formato brasileiro e carrego a base consolidada em `base`. A ideia é deixar a inspeção visual mais próxima da linguagem do desafio.

In [2]:
def formato_real(valor):
    if pd.isna(valor):
        return None

    return (
        f"{valor:,.2f}"
        .replace(",", "_")
        .replace(".", ",")
        .replace("_", ".")
    )


pd.set_option("display.float_format", formato_real)

base = pd.read_parquet("../dados_processados/finbra_consolidado.parquet")
base.head()

,Instituição,Cod.IBGE,UF,População,Coluna,Conta,Identificador da Conta,Valor,Ano,Tipo Conta
0,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,Despesas Exceto Intraorçamentárias,siconfi-cor_TotalDespesas,"874.885.274,98",2020,Totais Intraorçamentárias
1,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01 - Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",2020,Função
2,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01.031 - Ação Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",2020,Subfunção
3,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03 - Essencial à Justiça,siconfi-cor_TotalDespesas,"18.822.895,07",2020,Função
4,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03.092 - Representação Judicial e Extrajudicial,siconfi-cor_TotalDespesas,"18.822.895,07",2020,Subfunção


## Estrutura e tipagem

Aqui eu verifico se a base consolidada tem o formato esperado: colunas corretas, tipos coerentes e ausência de preenchimento indevido em campos-chave.

In [26]:
df = pd.read_parquet("../dados_processados/finbra_consolidado.parquet")

df.head()

,Instituição,Cod.IBGE,UF,População,Coluna,Conta,Identificador da Conta,Valor,Ano,Tipo Conta
0,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,Despesas Exceto Intraorçamentárias,siconfi-cor_TotalDespesas,"874.885.274,98",2020,Totais Intraorçamentárias
1,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01 - Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",2020,Função
2,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01.031 - Ação Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",2020,Subfunção
3,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03 - Essencial à Justiça,siconfi-cor_TotalDespesas,"18.822.895,07",2020,Função
4,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03.092 - Representação Judicial e Extrajudicial,siconfi-cor_TotalDespesas,"18.822.895,07",2020,Subfunção


## Completude por ano

O desafio avisa que 2025 não está completo. Esta checagem confirma isso e evita comparações injustas entre anos com cobertura diferente.

In [3]:
resumo_geral = pd.DataFrame(
    {
        "Métrica": ["Linhas", "Colunas", "Anos distintos", "Capitais distintas"],
        "Valor": [
            len(base),
            len(base.columns),
            base["Ano"].nunique(),
            base["Cod.IBGE"].nunique(),
        ],
    }
)

resumo_geral

,Métrica,Valor
0,Linhas,50334
1,Colunas,10
2,Anos distintos,6
3,Capitais distintas,26


In [4]:
estrutura_colunas = pd.DataFrame(
    {
        "Coluna": base.columns,
        "Tipo": base.dtypes.astype(str).values,
        "Nulos": base.isna().sum().values,
        "Não nulos": base.count().values,
    }
)

estrutura_colunas

,Coluna,Tipo,Nulos,Não nulos
0,Instituição,str,0,50334
1,Cod.IBGE,int64,0,50334
2,UF,str,0,50334
3,População,int64,0,50334
4,Coluna,str,0,50334
5,Conta,str,0,50334
6,Identificador da Conta,str,0,50334
7,Valor,float64,0,50334
8,Ano,int64,0,50334
9,Tipo Conta,str,0,50334


In [5]:
completude_anos = (
    base.groupby("Ano", as_index=False)
    .agg(
        capitais=("Cod.IBGE", "nunique"),
        registros=("Cod.IBGE", "size"),
    )
    .assign(
        status=lambda dados: dados["capitais"].apply(
            lambda total: "Completo" if total == 26 else "Parcial"
        ),
        capitais_faltantes=lambda dados: 26 - dados["capitais"],
    )
)

completude_anos

,Ano,capitais,registros,status,capitais_faltantes
0,2020,26,8988,Completo,0
1,2021,26,8867,Completo,0
2,2022,26,9312,Completo,0
3,2023,26,9576,Completo,0
4,2024,26,9528,Completo,0
5,2025,11,4063,Parcial,15


## Regras de negócio

A validação precisa ir além da estrutura: também é importante conferir se a base preserva as categorias esperadas do desafio, sem misturar totais com funções e subfunções.

In [6]:
estagios_despesas = (
    base.groupby("Coluna", as_index=False)
    .agg(
        registros=("Valor", "size"),
        valor_total=("Valor", "sum"),
    )
    .sort_values(by="valor_total", ascending=False)
)

estagios_despesas

,Coluna,registros,valor_total
0,Despesas Empenhadas,11495,"4.058.888.797.973,94"
1,Despesas Liquidadas,11435,"3.823.937.287.644,98"
2,Despesas Pagas,11403,"3.735.695.439.925,60"
3,Inscrição de Restos a Pagar Não Processados,8310,"231.594.623.309,91"
4,Inscrição de Restos a Pagar Processados,7691,"87.824.169.374,95"


In [7]:
tipos_conta = (
    base["Tipo Conta"]
    .value_counts()
    .rename_axis("Tipo Conta")
    .reset_index(name="Registros")
    .sort_values(by="Registros", ascending=False)
)

tipos_conta

,Tipo Conta,Registros
0,Subfunção,30122
1,Função,12813
2,Total Demais Subfunções,6095
3,Totais Intraorçamentárias,1304


## Conclusão

A validação indica que o ETL está consistente: a base consolidada abre corretamente, `Valor` está numérico, as colunas esperadas existem e os anos de 2020 a 2024 estão completos. O ano de 2025 segue parcial, então deve ser tratado como recorte incompleto.

Também ficou confirmado que as categorias do pipeline fazem sentido para a análise, com separação entre função, subfunção e linhas agregadas. Isso garante que o notebook de exploração possa trabalhar sobre uma base confiável sem precisar reclassificar as contas novamente.